        # 03 — Building the Embedding Index (Colab edition)

        **MIS 3080 Final Project — Movie Recommender**

        This notebook builds the **AI component** of the project: a dense vector representation of every movie's textual content, enabling semantic similarity search. A user can type a movie title, a genre name, or a free-form theme like *"dark sci-fi with a twist ending"* and we'll surface the movies whose plots match best.

        ### Where this fits in the architecture

        ```
        user query ──► [embed]  ──►  cosine sim vs. all movies  ──► top-50 candidates ──► [model.predict] re-rank ──► top-N shown to user
                       (this notebook)                                                    (notebook 02's model.joblib)
        ```

        The embeddings produced here are the **candidate generation** stage. Without them, we have no way to translate a user's natural-language query into a list of relevant movies — the rating regressor alone can rank movies but cannot find them.

        ### Approach

        We use **`sentence-transformers/all-MiniLM-L6-v2`** — a 22M-parameter model that maps any sentence (or short paragraph) to a 384-dimensional unit vector. After normalization, **cosine similarity reduces to a dot product**, so search is a single matrix multiply.

        Why this model:
        - **Free, local, no API key** — important for both privacy and grading reproducibility.
        - **Small (~80 MB)** — loads in ~1 second, encodes the entire 4,803-movie corpus in well under a minute on CPU.
        - **Strong general-purpose semantic similarity** — trained on >1B sentence pairs, widely used as a default in the community.

        ### What this notebook produces

        - **`embeddings.npy`** — a `(4803, 384)` `float32` numpy array. Row `i` is the L2-normalized embedding for the movie at row `i` of `movies_clean.parquet`. Saved size: ~7 MB.


### Running this notebook in Google Colab

1. Run `01_eda_colab.ipynb` first to produce `movies_clean.parquet`. Save it locally.
2. Click **Runtime → Run all** in this notebook.
3. When prompted, upload `movies_clean.parquet`.
4. **Optional but ~10× faster**: switch the runtime to GPU (**Runtime → Change runtime type → T4 GPU**) before running. The encoder will auto-detect it. CPU works fine — just slower.
5. The final cell auto-downloads `embeddings.npy` to your computer. Keep that file — `app.py` loads it at startup.


## 0. Setup — Colab detection, dependencies, and data upload


In [ ]:
# Colab pre-installs torch and pandas. We just need sentence-transformers.
%pip install -q sentence-transformers pyarrow


In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
print("running in Colab :", IN_COLAB)


### Upload `movies_clean.parquet`

When the next cell runs in Colab, a file picker will appear. Upload the parquet you produced (and downloaded) from `01_eda_colab.ipynb`.


In [ ]:
from pathlib import Path

if IN_COLAB:
    from google.colab import files
    uploaded = files.upload()  # expects movies_clean.parquet
    DATA_DIR = Path("/content")
    for name in uploaded:
        print(f"uploaded {name} ({len(uploaded[name])/1024:.1f} KB)")
else:
    DATA_DIR = Path.cwd().parent / "artifacts" if Path.cwd().name == "notebooks" else Path.cwd() / "artifacts"

DATA_PATH = DATA_DIR / "movies_clean.parquet"
ARTIFACTS = Path("/content") if IN_COLAB else DATA_DIR
ARTIFACTS.mkdir(parents=True, exist_ok=True)

assert DATA_PATH.exists(), f"missing {DATA_PATH} - upload movies_clean.parquet first"
print("DATA_PATH:", DATA_PATH)
print("ARTIFACTS:", ARTIFACTS)


In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer

np.random.seed(42)


## 1. Load the cleaned dataset


We use `movies_clean.parquet` straight from notebook 01. Important: we keep **all 4,803 movies** in the embedding index (not just `train_eligible`) — a user should be able to discover obscure films too. The rating model will quietly down-rank low-vote movies later via predicted-quality scoring, which is the appropriate place to make that decision.


In [ ]:
df = pd.read_parquet(DATA_PATH).reset_index(drop=True)
print("rows:", len(df))
print("columns w/ embed_text:", df["embed_text"].notna().sum())
print()
print("sample embed_text:")
print(df["embed_text"].iloc[0][:300])


In [ ]:
# Length distribution of the corpus we'll be embedding
lengths = df["embed_text"].fillna("").str.len()
fig, ax = plt.subplots(figsize=(8, 3.4))
lengths.plot.hist(bins=50, ax=ax, edgecolor="white")
ax.set_title("Length (chars) of embed_text per movie")
ax.set_xlabel("characters")
plt.show()
print("median chars:", int(lengths.median()), "  max:", int(lengths.max()))


**Note on truncation:** `all-MiniLM-L6-v2` truncates input at 256 tokens (~1,000 characters of typical English). A small handful of movies with very long keyword lists exceed this. The truncated embedding still captures the title + tagline + overview, which is the most informative content. We don't worry about it.


## 2. Load the encoder


First call downloads the model (~80 MB) and caches it. Subsequent loads are instant. The model auto-detects GPU if available — on Colab with a T4 enabled, encoding takes ~5 seconds; on CPU, ~40 seconds.


In [ ]:
t0 = time.time()
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print(f"loaded in {time.time()-t0:.1f}s")
print("embedding dim:", model.get_sentence_embedding_dimension())
print("device       :", model.device)


## 3. Encode the corpus


We encode in batches of 64 with `normalize_embeddings=True` so that **every output vector has L2 norm = 1**. This means cosine similarity between two vectors equals their dot product — search becomes a single matrix multiply later.

Replacing any NaN `embed_text` with an empty string is a defensive guard against malformed rows.


In [ ]:
texts = df["embed_text"].fillna("").tolist()

t0 = time.time()
embeddings = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True,
)
elapsed = time.time() - t0
print(f"encoded {len(texts):,} texts in {elapsed:.1f}s "
      f"({len(texts)/elapsed:.0f} texts/sec)")

embeddings = embeddings.astype(np.float32)  # save space; precision is plenty
print("matrix shape:", embeddings.shape)
print("dtype       :", embeddings.dtype)
print("norm of row 0 (should be ~1.0):", float(np.linalg.norm(embeddings[0])))


## 4. Save the embedding matrix


In [ ]:
out_path = ARTIFACTS / "embeddings.npy"
np.save(out_path, embeddings)
print(f"saved {out_path} ({out_path.stat().st_size/1024:.1f} KB)")


In [ ]:
# Roundtrip sanity check
reloaded = np.load(out_path)
assert reloaded.shape == embeddings.shape
assert np.allclose(reloaded, embeddings)
print("roundtrip OK")


### Download `embeddings.npy` (Colab)

Triggers a browser download so you can save the file for the Streamlit app. The download skips automatically when running outside Colab.


In [ ]:
if IN_COLAB:
    from google.colab import files
    files.download(str(out_path))
else:
    print("Local run - file already at", out_path)


## 5. Search demo — does it actually work?


We define a small `search(query, k)` helper that:

1. Encodes the query (also normalized, same as the corpus).
2. Computes the dot product against every movie embedding (vectorized — runs in milliseconds).
3. Returns the top-`k` movies sorted by similarity.

This is exactly the **candidate generation** logic that will run inside the Streamlit app — no surprises at deploy time.


In [ ]:
def search(query: str, k: int = 10) -> pd.DataFrame:
    qv = model.encode([query], normalize_embeddings=True)[0]
    sims = embeddings @ qv  # (4803,) cosine similarities
    top_idx = np.argsort(-sims)[:k]
    cols = ["title", "primary_genre", "release_year", "vote_average"]
    return df.iloc[top_idx][cols].assign(similarity=sims[top_idx].round(3))


### A few exploratory queries


In [ ]:
search("dark sci-fi with a twist ending", k=10)


In [ ]:
search("heartwarming animated movie about friendship", k=10)


In [ ]:
search("war drama set in vietnam", k=10)


In [ ]:
search("revenge thriller with a female lead", k=10)


### Search by movie title — the user's most common pattern

In [ ]:
search("The Dark Knight", k=10)


In [ ]:
search("Inception", k=10)


### What you should see

- **Title queries** return the movie itself first (similarity ~0.6-0.9), then thematic neighbors (sequels, franchise members, similar plots).
- **Theme queries** surface movies whose plot/keyword text matches the query semantics — even when the user's wording never appears verbatim in the movie's overview.
- **Some queries are imperfect** — for example "heartwarming animated movie about friendship" tends to over-weight "heartwarming" sentiment vs. "animated" — this is the limitation of a general-purpose embedding model. The rating-based re-rank in `app.py` doesn't fix this directly, but the similarity score will downweight obviously-off matches.

For the writeup: this is a **good** thing to show — semantic embeddings are powerful but imperfect, and the project's value comes from combining them with the rating predictor, not relying on either alone.


## 6. Preview: how this combines with the rating regressor


Just to sanity-check that everything fits together, we'll load `model.joblib` from notebook 02 and run the full pipeline on a single query — exactly what the Streamlit app will do at request time.


In [ ]:
import joblib
try:
    ranker = joblib.load(ARTIFACTS / "model.joblib")
    have_ranker = True
    print("loaded model.joblib")
except FileNotFoundError:
    have_ranker = False
    print("model.joblib not found - run notebook 02 first to enable this preview")


In [ ]:
if have_ranker:
    FEATURE_COLS = [
        "release_year", "runtime",
        "log_budget", "log_revenue", "log_popularity", "log_vote_count",
        "n_genres", "n_keywords", "cast_size",
        "primary_genre", "original_language",
        "director", "lead_actor",
    ]

    def recommend(query: str, candidate_k: int = 50, final_k: int = 10,
                  sim_weight: float = 0.6, rating_weight: float = 0.4):
        # 1) Candidate generation via embeddings
        qv = model.encode([query], normalize_embeddings=True)[0]
        sims = embeddings @ qv
        cand_idx = np.argsort(-sims)[:candidate_k]
        cand = df.iloc[cand_idx].copy()
        cand["similarity"] = sims[cand_idx]

        # 2) Rating prediction on the candidates
        cand["pred_rating"] = ranker.predict(cand[FEATURE_COLS])

        # 3) Blend (normalize each to 0..1 first)
        sim_n  = (cand["similarity"]  - cand["similarity"].min())  / (cand["similarity"].max()  - cand["similarity"].min()  + 1e-9)
        rate_n = (cand["pred_rating"] - cand["pred_rating"].min()) / (cand["pred_rating"].max() - cand["pred_rating"].min() + 1e-9)
        cand["score"] = sim_weight * sim_n + rating_weight * rate_n

        return cand.nlargest(final_k, "score")[
            ["title", "primary_genre", "release_year",
             "vote_average", "similarity", "pred_rating", "score"]
        ].round(3)

    recommend("dark sci-fi with a twist ending", final_k=10)


In [ ]:
if have_ranker:
    display(recommend("heartwarming animated movie about friendship", final_k=10))
    display(recommend("war drama set in vietnam", final_k=10))


**What's happening above:** the embeddings pull in 50 thematically similar candidates; the rating regressor scores each one's expected quality; the final order is a blend (60% similarity, 40% predicted rating).

Note how poorly-rated movies that *do* match the theme get pushed down by the rating component — exactly what the project rubric calls a "useful, actionable result." The exact blend weights are a knob `app.py` will expose to the user.


## 7. Next steps

With `embeddings.npy` saved alongside `model.joblib` and `movies_clean.parquet`, the artifacts directory now has everything `app.py` needs:

```
artifacts/
├── movies_clean.parquet   ← row order matches embeddings.npy
├── embeddings.npy         ← (4803, 384) float32, L2-normalized
├── model.joblib           ← fitted Pipeline(prep + XGBoost)
└── model_meta.json        ← metrics + feature lists for the writeup
```

Up next: **`app.py`** — a Streamlit interface that loads all four artifacts at startup, accepts a free-form query, and returns ranked recommendations with explanations.
